[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyMongo and Beanie, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)

# Update Operators &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's boot cell, with `fresh_item` and `item`. Run it first. Every task
below calls `fresh_item()` before it starts, so they can be run in any order and as often as you
like.


In [1]:
import os
import random
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("pymongo") != "4.18.1" or version("beanie") != "2.2.0":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "pymongo==4.18.1", "beanie==2.2.0"], check=True)

import pymongo
from pymongo import ReturnDocument

DBPATH = "/content/mongo" if os.path.isdir("/content") else "/tmp/guide_mongo/rs"
LOGPATH = f"{DBPATH}.log"
URI = "mongodb://127.0.0.1:27017/shop"                              # no credential, anywhere
PUBLISHED = ["jammy", "noble"]                                      # codenames MongoDB builds for


def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(timeout=2000):
    """Whether a mongod is there, asked directly rather than through topology discovery."""
    try:
        with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                                 serverSelectionTimeoutMS=timeout) as client:
            client.admin.command("ping")
            return True
    except pymongo.errors.PyMongoError:
        return False


def install_server():
    """Add MongoDB's own apt repository and install the server package. Linux only."""
    if shell("which mongod")[0] == 0:
        return "already installed"

    codename = shell("lsb_release -cs")[1]
    if codename not in PUBLISHED:                                   # an unpublished one breaks apt
        print(f"  Ubuntu '{codename}' has no MongoDB repository; using '{PUBLISHED[-1]}' instead")
        codename = PUBLISHED[-1]

    if not shell("grep -o avx /proc/cpuinfo | head -1")[1]:
        raise RuntimeError("This CPU has no AVX. Every MongoDB build since 5.0 needs it, so "
                           "neither the apt package nor the tarball will start here.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"curl -fsSL https://www.mongodb.org/static/pgp/server-8.0.asc "
          f"| {sudo}gpg --dearmor -o /usr/share/keyrings/mongodb-8.0.gpg")
    shell(f'echo "deb [signed-by=/usr/share/keyrings/mongodb-8.0.gpg] '
          f'https://repo.mongodb.org/apt/ubuntu {codename}/mongodb-org/8.0 multiverse" '
          f'| {sudo}tee /etc/apt/sources.list.d/mongodb-8.0.list')
    shell(f"{sudo}apt-get -qq update "                              # this one list file only
          f"-o Dir::Etc::sourcelist=sources.list.d/mongodb-8.0.list "
          f"-o Dir::Etc::sourceparts=-")
    code, out = shell(f"{sudo}apt-get -qq -y install mongodb-org-server")
    if shell("which mongod")[0] != 0:
        raise RuntimeError(f"mongodb-org-server did not install. apt said: {out[-400:]}")
    return f"installed from the {codename} repository"


def start_server(wait=30):
    """Start mongod with a replica set name, idempotently. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No mongod is answering on 127.0.0.1:27017. Start your own server "
                           "with --replSet rs0 and run this again: this cell only installs one "
                           "on Linux, which is what Colab runs.")

    print(" ", install_server())
    os.makedirs(DBPATH, exist_ok=True)
    code, out = shell(f"mongod --dbpath {DBPATH} --replSet rs0 --bind_ip 127.0.0.1 "
                      f"--fork --logpath {LOGPATH}")
    if code != 0:                                                   # --fork hides the reason
        print("  mongod did not start. The last lines of its log:")
        print("   ", shell(f"tail -20 {LOGPATH}")[1].replace("\n", "\n    "))
        raise RuntimeError("mongod exited. The log above says why.")

    for attempt in range(1, wait + 1):
        if answering():
            return "installed and started"
        print(f"  waiting for mongod ({attempt})")
        time.sleep(1)
    raise RuntimeError(f"mongod did not answer within {wait} seconds.")

def initiate(wait=30):
    """Make the single node a replica set, which is what transactions and migrations need."""
    with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                             serverSelectionTimeoutMS=2000) as boot:
        try:                                                        # an explicit host, not getHostName()
            boot.admin.command("replSetInitiate",
                               {"_id": "rs0", "members": [{"_id": 0, "host": "127.0.0.1:27017"}]})
        except pymongo.errors.OperationFailure as error:
            if error.code != 23:                                    # 23 is AlreadyInitialized
                raise

        for attempt in range(1, wait + 1):
            hello = boot.admin.command("hello")
            if hello.get("isWritablePrimary"):
                return f"replica set {hello['setName']}, primary"
            time.sleep(1)
    raise RuntimeError(f"No primary after {wait} seconds. The last hello was: {hello}")

SIZE = 500                                                          # Indexes and the catalog raise this
KINDS = ["laptop", "monitor", "keyboard", "mouse", "cable"]
MAKERS = ["Aster", "Belden", "Corvid", "Dalgo"]


def seed(size=None, force=False):
    """Fill shop.products and shop.reviews, once, from a fixed seed so every run agrees."""
    size = SIZE if size is None else size
    client = pymongo.MongoClient(URI, tz_aware=True)
    shop = client.get_default_database()

    if not force and shop.products.estimated_document_count() == size:
        client.close()
        return size

    shop.products.drop()
    shop.reviews.drop()
    random.seed(0)                                                  # the whole reason runs agree

    products, reviews = [], []
    for number in range(size):
        kind = KINDS[number % len(KINDS)]
        product = {
            "_id": number,
            "sku": f"{kind[:3].upper()}-{number:06d}",
            "name": f"{MAKERS[number % len(MAKERS)]} {kind} {number}",
            "maker": MAKERS[number % len(MAKERS)],
            "kind": kind,
            "price": round(random.uniform(5, 2000), 2),
            "stock": random.randint(0, 400),
            "tags": sorted(random.sample(["sale", "new", "refurbished", "bulk", "clearance"], 2)),
            "size": {"w": random.randint(5, 60), "h": random.randint(2, 40)},
        }
        products.append(product)
        for _ in range(random.randint(0, 3)):
            reviews.append({"product_id": number, "stars": random.randint(1, 5),
                            "body": f"A review of {product['name']}"})

    for start in range(0, len(products), 5000):                     # batches, not one huge insert
        shop.products.insert_many(products[start:start + 5000])
    for start in range(0, len(reviews), 5000):
        shop.reviews.insert_many(reviews[start:start + 5000])

    client.close()
    return size


def report():
    """One line naming what this notebook is running against."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        build = client.admin.command("buildInfo")["version"].split(".")[0]
        shop = client.get_default_database()
        return (f"MongoDB {build} | pymongo {version('pymongo')} | beanie {version('beanie')} "
                f"| products: {shop.products.count_documents({})}")

def failed(error):
    """A write or command failure's real message, without the parts that change every run."""
    details = getattr(error, "details", None) or {}
    return f"{type(error).__name__}: {details.get('errmsg', str(error).split(', full error')[0])}"


def fresh_item():
    """One document to write on, put back exactly as it was before each section."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        shop = client.get_default_database()
        shop.items.delete_many({"_id": 1})
        shop.items.insert_one({
            "_id": 1,
            "name": "keyboard",
            "price": 50,
            "stock": 4,
            "tags": ["sale"],
            "lines": [{"sku": "a", "qty": 1}, {"sku": "b", "qty": 2}],
        })
        return shop.items.find_one({"_id": 1})


def item():
    """What is in it now."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        return client.get_default_database().items.find_one({"_id": 1})


print("server: ", start_server())
print("replica:", initiate())
print("seeded: ", seed(), "products")
print("item:   ", sorted(fresh_item()))
print(report())


server:  already running
replica: replica set rs0, primary
seeded:  500 products
item:    ['_id', 'lines', 'name', 'price', 'stock', 'tags']
MongoDB 8 | pymongo 4.18.1 | beanie 2.2.0 | products: 500


**1.** One field, and only one field.


In [2]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()
fresh_item()

print("before:", sorted(item()), "| price", item()["price"])
shop.items.update_one({"_id": 1}, {"$set": {"price": 75}})
print("after: ", sorted(item()), "| price", item()["price"])
client.close()


before: ['_id', 'lines', 'name', 'price', 'stock', 'tags'] | price 50
after:  ['_id', 'lines', 'name', 'price', 'stock', 'tags'] | price 75


`$set` names the field it changes and leaves everything else alone. That is the whole difference
from the next task.


**2.** The whole document, and what is left of it.


In [3]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()
fresh_item()

print("before:", sorted(item()))
shop.items.replace_one({"_id": 1}, {"name": "keyboard", "price": 75})
print("after: ", sorted(item()))
print("lost:  ", sorted(set(fresh_item()) - {"_id", "name", "price"}))
client.close()


before: ['_id', 'lines', 'name', 'price', 'stock', 'tags']
after:  ['_id', 'name', 'price']
lost:   ['lines', 'stock', 'tags']


Three fields gone, one modified document reported, no error. `replace_one` did exactly what it
promises, which is the problem.


**3.** Arithmetic on the server.


In [4]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()
fresh_item()

print("stock before:", item()["stock"])
shop.items.update_one({"_id": 1}, {"$inc": {"stock": -1}})
print("stock after: ", item()["stock"])
print("the value was never in Python, so nothing could have raced with it")
client.close()


stock before: 4
stock after:  3
the value was never in Python, so nothing could have raced with it


Reading the number, subtracting one and writing it back would be two round trips. Two processes
doing that at once both read four and both write three, and one sale disappears.


**4.** Two ways to append.


In [5]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()

fresh_item()
shop.items.update_one({"_id": 1}, {"$push": {"tags": "bulk"}})
shop.items.update_one({"_id": 1}, {"$push": {"tags": "bulk"}})
print("$push twice:     ", item()["tags"])

fresh_item()
shop.items.update_one({"_id": 1}, {"$addToSet": {"tags": "bulk"}})
shop.items.update_one({"_id": 1}, {"$addToSet": {"tags": "bulk"}})
print("$addToSet twice: ", item()["tags"])
client.close()


$push twice:      ['sale', 'bulk', 'bulk']
$addToSet twice:  ['sale', 'bulk']


`$push` is a list operation and `$addToSet` is a set operation. Retrying a failed request is the
usual reason a `$push` runs twice, which is why `$addToSet` is the safer default for tags.


**5.** What an upsert builds.


In [6]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()
shop.answers.drop()

shop.answers.update_one({"sku": "AAA-1", "region": "eu"},
                        {"$set": {"count": 3}}, upsert=True)
print("built:", shop.answers.find_one({}, {"_id": 0}))
print("sku and region came from the filter, count came from the update")
client.close()


built: {'sku': 'AAA-1', 'region': 'eu', 'count': 3}
sku and region came from the filter, count came from the update


Every equality condition in the filter becomes a field of the new document. A condition written with
an operator contributes nothing, because there is no single value to use.


**6.** Before and after.


In [7]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()
fresh_item()

old = shop.items.find_one_and_update({"_id": 1}, {"$inc": {"stock": 10}})
print("default returned:", old["stock"], "| stored now:", item()["stock"])

new = shop.items.find_one_and_update({"_id": 1}, {"$inc": {"stock": 10}},
                                     return_document=ReturnDocument.AFTER)
print("AFTER returned:  ", new["stock"], "| stored now:", item()["stock"])
client.close()


default returned: 4 | stored now: 14
AFTER returned:   24 | stored now: 24


The default is `BEFORE`. It is the right choice when you are claiming something and want the state
you claimed it in, and the wrong choice every other time.


---

&#8592; **Back to:** [Update Operators](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/06-update-operators.ipynb)  &nbsp;&middot;&nbsp;  [PyMongo and Beanie, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)
